# Tokenize DPO Dataset
Reads raw preference pairs from `datasets_dpo/<dataset>/`, tokenizes chosen and rejected
sequences with prefix boundary preserved, and writes tokenized JSONL.

Reference logits are NOT computed here — they are cached in the DPO training notebook
so you only need to re-run training when iterating on RL.

Run `download_ultrafeedback` first. Run `dpo_ultrafeedback` after.

In [ ]:
import os
import json
from google.colab import drive
from tokenizers import Tokenizer

drive.mount('/content/drive')

SPARKYLLM = "/content/drive/MyDrive/sparkyllm"
TOKENIZER_PATH = os.path.join(SPARKYLLM, "datasets_pretrain", "tokenizer_out", "tokenizer.json")

# ==================== CONFIGURATION ====================
DATASET_NAME = "ultrafeedback"  # change for other DPO datasets
BLOCK_SIZE = 768
MIN_RESPONSE_TOKENS = 3

DPO_DIR = os.path.join(SPARKYLLM, "datasets_dpo", DATASET_NAME)
RAW_FILE = os.path.join(DPO_DIR, f"{DATASET_NAME}_raw.jsonl")
OUT_FILE = os.path.join(DPO_DIR, f"{DATASET_NAME}_tokenized.jsonl")
META_FILE = os.path.join(DPO_DIR, "tokenize_meta.json")

assert os.path.exists(RAW_FILE), f"Raw file not found: {RAW_FILE}\nRun download_{DATASET_NAME} first."

# Load tokenizer
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
eot_id = tokenizer.token_to_id("<|endoftext|>")
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer: {vocab_size} tokens | EOT: {eot_id}")
print(f"Raw file: {RAW_FILE}")

In [ ]:
# ==================== FORMAT ADAPTER ====================
# Each DPO dataset needs a function that takes a raw JSON object
# and returns (prefix_text, chosen_text, rejected_text).
# Add new adapters here for other datasets.

def ultrafeedback_adapter(obj):
    """UltraFeedback: prompt -> chosen/rejected responses."""
    prefix_text = f"Instruction: {obj['prompt']}\nResponse: "
    return prefix_text, obj["chosen"], obj["rejected"]

ADAPTERS = {
    "ultrafeedback": ultrafeedback_adapter,
}

adapter = ADAPTERS[DATASET_NAME]
print(f"Using adapter: {DATASET_NAME}")

In [ ]:
# ==================== TOKENIZE ====================

written = 0
skipped = 0
total_chosen_tokens = 0
total_rejected_tokens = 0
total_prefix_len = 0

with open(OUT_FILE, "w", encoding="utf-8") as f_out:
    with open(RAW_FILE, "r", encoding="utf-8") as f_in:
        for line_num, line in enumerate(f_in):
            obj = json.loads(line)
            prefix_text, chosen_text, rejected_text = adapter(obj)

            prefix_ids = tokenizer.encode(prefix_text).ids
            chosen_resp_ids = tokenizer.encode(chosen_text).ids
            rejected_resp_ids = tokenizer.encode(rejected_text).ids

            if len(chosen_resp_ids) < MIN_RESPONSE_TOKENS or len(rejected_resp_ids) < MIN_RESPONSE_TOKENS:
                skipped += 1
                continue

            # Full sequences: prefix + response + EOT
            chosen_ids = prefix_ids + chosen_resp_ids + [eot_id]
            rejected_ids = prefix_ids + rejected_resp_ids + [eot_id]

            # Truncate to block_size + 1, preserving EOT at the end
            if len(chosen_ids) > BLOCK_SIZE + 1:
                chosen_ids = chosen_ids[:BLOCK_SIZE] + [eot_id]
            if len(rejected_ids) > BLOCK_SIZE + 1:
                rejected_ids = rejected_ids[:BLOCK_SIZE] + [eot_id]

            out_obj = {
                "chosen_ids": chosen_ids,
                "rejected_ids": rejected_ids,
                "prefix_len": len(prefix_ids),
            }
            f_out.write(json.dumps(out_obj) + "\n")

            written += 1
            total_chosen_tokens += len(chosen_ids)
            total_rejected_tokens += len(rejected_ids)
            total_prefix_len += len(prefix_ids)

            if (line_num + 1) % 10000 == 0:
                print(f"  {line_num + 1:,} lines processed...")

avg_prefix = total_prefix_len / written if written else 0
avg_chosen = total_chosen_tokens / written if written else 0
avg_rejected = total_rejected_tokens / written if written else 0

print(f"\nDone!")
print(f"  {written:,} pairs written, {skipped:,} skipped")
print(f"  Avg prefix: {avg_prefix:.1f} | Avg chosen: {avg_chosen:.1f} | Avg rejected: {avg_rejected:.1f} tokens")
print(f"  Saved to: {OUT_FILE}")

# Save metadata
meta = {
    "dataset": DATASET_NAME,
    "num_pairs": written,
    "skipped": skipped,
    "total_chosen_tokens": total_chosen_tokens,
    "total_rejected_tokens": total_rejected_tokens,
    "avg_prefix_len": round(avg_prefix, 1),
    "avg_chosen_len": round(avg_chosen, 1),
    "avg_rejected_len": round(avg_rejected, 1),
    "block_size": BLOCK_SIZE,
    "eot_id": eot_id,
    "vocab_size": vocab_size,
    "tokenizer_path": TOKENIZER_PATH,
}
with open(META_FILE, "w") as f:
    json.dump(meta, f, indent=2)
print(f"  Meta: {META_FILE}")